In [3]:
import pandas as pd
from bparadigm.paths import REDDIT_TOPICS, PROCESSED

In [4]:
enriched = pd.read_csv(REDDIT_TOPICS)

def brand_of(r):
    if r["mentions_apple"] and r["mentions_samsung"]: return "Both"
    if r["mentions_apple"]:   return "Apple"
    if r["mentions_samsung"]: return "Samsung"

enriched["brand"] = enriched.apply(brand_of, axis=1)
enriched.to_csv(PROCESSED / "reddit_enriched.csv", index=False)

print(enriched.shape)
print(enriched["brand"].value_counts())
print(enriched["sentiment"].value_counts())

(2453, 14)
brand
Samsung    1243
Apple      1090
Both        120
Name: count, dtype: int64
sentiment
positive    1349
negative    1104
Name: count, dtype: int64


In [5]:
scored = enriched[enriched["sentiment"].notna()]
print(scored.groupby("brand")["sentiment"]
      .apply(lambda s: round((s == "positive").mean() * 100, 1)))

brand
Apple      53.8
Both       57.5
Samsung    55.8
Name: sentiment, dtype: float64


In [6]:
cross = [2, 9, 14, 24, 34, 36]
sub = scored[(scored["topic"].isin(cross)) & (scored["brand"].isin(["Apple", "Samsung"]))]

kpi = (sub.groupby(["topic_name", "brand"])
       .agg(n=("sentiment", "size"),
            pct_pos=("sentiment", lambda s: round((s == "positive").mean() * 100, 1)))
       .reset_index())
print(kpi.to_string(index=False))

                               topic_name   brand  n  pct_pos
              Android navigation gestures   Apple 14     28.6
              Android navigation gestures Samsung 11     18.2
                        Charging & cables   Apple 18     44.4
                        Charging & cables Samsung 24     33.3
            OnePlus 6T vs Galaxy / iPhone   Apple  6     83.3
            OnePlus 6T vs Galaxy / iPhone Samsung 11     63.6
Smartwatches: Galaxy Watch vs Apple Watch   Apple 18     83.3
Smartwatches: Galaxy Watch vs Apple Watch Samsung 36     58.3
     Social app camera quality on Android   Apple  6     16.7
     Social app camera quality on Android Samsung  7     14.3
          Switching phones between brands   Apple 53     37.7
          Switching phones between brands Samsung 36     47.2


In [7]:
by_topic = (scored[scored["topic_name"].notna()]
            .groupby("topic_name")
            .agg(n=("sentiment", "size"),
                 pct_pos=("sentiment", lambda s: round((s == "positive").mean() * 100, 1)))
            .query("n >= 25")
            .sort_values("pct_pos"))
print(by_topic.to_string())

                                             n  pct_pos
topic_name                                             
Apple Store support & AppleCare             28     10.7
Samsung TVs & home AV                       33     27.3
Android navigation gestures                 28     28.6
Galaxy S10 fingerprint sensor & security    32     31.2
Android Pie / One UI update problems        73     38.4
Galaxy Buds preorder & stock                55     40.0
Charging & cables                           45     40.0
Switching phones between brands             98     40.8
Apple Music & streaming subscriptions       42     42.9
Galaxy Buds audio problems                  93     43.0
Apple–Qualcomm legal settlement             29     44.8
Screen protection & phone durability        39     46.2
MacBook keyboard & hardware                 49     49.0
Apple Card & Apple Pay                      29     51.7
Apple services: News+, TV+, Arcade          40     52.5
Galaxy Fold launch & delay                 162  

In [9]:
from bparadigm.paths import PROCESSED
(scored.groupby("brand")
   .agg(n=("sentiment", "size"),
        pct_positive=("sentiment", lambda s: round((s == "positive").mean() * 100, 1)))
   .reset_index()
   .query("brand in ['Apple','Samsung']")
   .to_csv(PROCESSED / "brand_results.csv", index=False))
print("saved brand_results.csv")

saved brand_results.csv
